# Automated Motor Switching

## Libraries

In [1]:
import serial

import pandas as pd

import time
from watchdog.observers import Observer
from watchdog.events import FileSystemEventHandler

## Experimental Parameters

The following parameters can be adapted depending on the experimental setup and the computer used.

In [ ]:
PORT = "COM3"  # Serial port connected to the Arduino (e.g. "COMX" on Windows or "/dev/cu.usbmodemXXXX" on Mac)
BAUDRATE = 19200  # Communication speed (1/s); must match to Arduino baudrate

WAITING_TIME = 5  # Waiting time after trigger (s)
WAVELENGTH = 500  # Wavelength to monitor (nm)
THRESHOLD = 250  # Intensity threshold for triggering (a.u.)

WATCH_DIRECTORY = r"C:\Path\To\Monitored\Folder"  # Path to folder that will be monitored for new spectra

## Functions and Definitions

In [ ]:
# Definition of a serial object representing the Arduino
dev = serial.Serial(port=PORT, baudrate=BAUDRATE)

# Definition of global variables 
triggered = False  # False if intensity is above 'threshold', True if value is below 'threshold'
last_trigger_time = 0  # Time point of last trigger 
wait = WAITING_TIME  # Waiting time after trigger (s)

This following function opens an UV/Vis spectrum and compares the intensity at an monitored wavelength to a defined 'threshold' intensity. If 'threshold' is crossed, a signal ('0' or '1') is transmitted to the Arduino that eventually switches the motor. <br>
The spectral data is subject to a certain amount of noise. Slowly crossing 'threshold' in combination with the noise can thus lead to incorrect transmission. For this reason, there is wait ('WAITING_TIME') after each switch before further checks are performed.

In [ ]:
def check_wavelength(filepath, for_wavelength, threshold):
    """
    Check the intensity at a specific wavelength in a measurement file and trigger the Arduino if thresholds are crossed.

    Parameters
    ----------
    filepath : str
        Path to the txt-file containing spectral data.
    for_wavelength : float
        Wavelength to monitor.
    threshold : int
        Intensity threshold for triggering.
    """

    global triggered
    global last_trigger_time
    global wait

    data = pd.read_csv(filepath, skiprows=5, sep=";")

    # Read parameters
    wavelength = data.iloc[1:,0].str.replace(",", ".").astype(float)
    intensity = data.iloc[1:,1].str.replace(",", ".").astype(float)
    
    idx = (wavelength - for_wavelength).abs().idxmin()  # Find index of monitored wavelength

    now = time.time()  # Get current time to compare to 'last_trigger_time'


    # Check the intensity and send signal to Arduino if threshold is crossed.
    # Then wait for the defined cooldown period.
    if now - last_trigger_time < wait:
        # Check if 'wait' time has already passed since last trigger.
        print(f"Waiting, second {now-last_trigger_time}")

    elif intensity[idx] <= threshold and triggered == True:
        # Intensity is below threshold and 'triggered' is set to below.
        print("Intensity below threshold")

    elif intensity[idx] <= threshold and triggered == False:
        # Intensity is below threshold, but 'triggered' is set to above.
        # -> Intensity just crossed below the threshold.
        print("Intensity just fell below threshold.")
        dev.write(b"1")  # Send signal to Arduino
        triggered = True
        last_trigger_time = now  # Start cooldown period

    elif intensity[idx] > threshold and triggered == False:
        # Intensity is above threshold and 'triggered' is set to above 
        print("Intensity above threshold")

    elif intensity[idx] > threshold and triggered == True:
        # Intensity is above threshold, but 'triggered' is set to below.
        # -> Intensity just crossed above the threshold.
        print("Intensity just exceeded threshold")
        dev.write(b"0")  # Send signal to Arduino
        triggered = False
        last_trigger_time = now  # Start cooldown period

    else:
        # Unexpected case
        print("Something went wrong")

The following classes are definied to monitor the folder where the UV/Vis spectra are continuously stored.

In [ ]:
# Class: OnMyWatch
# Purpose: Set up and run a folder observer that watches for new files.
class OnMyWatch:

    # Set the directory on watch.
    watchDirectory = WATCH_DIRECTORY

    def __init__(self):
        # Create an Observer object from watchdog.
        self.observer = Observer()

    def run(self):
        """
        Start monitoring the folder and handle events using the Handler class (see below).
        The observer runs indefinitely until interrupted.
        """

        event_handler = Handler()
        self.observer.schedule(event_handler, self.watchDirectory, recursive = True)
        self.observer.start()

        try:
            while True:
                # Sleep to reduce CPU usage.
                time.sleep(5)
        except:
            # Stop observer if interrupted manually.
            self.observer.stop()
            print("Observer Stopped")

        self.observer.join()


#--------------------------------------------------------------
# Class: Handler
# Purpose: Handle events when files are created in the folder.
class Handler(FileSystemEventHandler):

    @staticmethod
    def on_any_event(event):
        """
        Called on any filesystem event.
        Processes only files (ignores directories) and triggers processing when a new file is created.
        """

        if event.is_directory:
            return None  # Ignore directories

        elif event.event_type == 'created':
            # A new file has been created and can now be processed.

            time.sleep(1)  # Wait to ensure the file is fully written.

            # Now the previously implemented function is called.
            # The 'filepath' of the newly created file is set using 'event.src_path'.
            # The monitored wavelength and threshold can be adjusted here.
            check_wavelength(event.src_path, WAVELENGTH, THRESHOLD)

## Main Part

Run the folder monitoring using the previously defined classes and functions. <br>
This block starts the observer and keeps it running until manually stopped.

In [ ]:
if __name__ == '__main__':
    watch = OnMyWatch()
    watch.run()